### Notebook 13 — model_registration

##### 1. Purpose

Register the successfully tracked Transformer model from MLflow into Unity Catalog Model Registry, then verify that the registered version can be loaded and used for raw-text inference.

The flow is:

``` text

Notebook 12 Logged Model
models:/m-xxxxxxxx
        ↓
Unity Catalog Model Registry
        ↓
dbw_agentic_ai_dev
.support_ticket_ai
.support_ticket_transformer_classifier
        ↓
Version 1 / Version N
        ↓
Load registered version
        ↓
Raw-text inference

```

No Transformer retraining happens in this notebook.


##### Architecture:

``` text

Find experiment
   ↓
Find successful run
   ↓
Find READY Logged Model
   ↓
Build source_model_uri
   ↓
mlflow.models.get_model_info(source_model_uri)
   ↓
model_info
   ↓
Check model_info.signature
   ↓
Register model

```

##### 2. Imports

In [0]:
import mlflow
import pandas as pd

from mlflow import MlflowClient

from src.project_config import (
    CLASS_NAMES,
    TEXT_COL,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_RUN_NAME,
    REGISTERED_MODEL_NAME,
    REGISTERED_MODEL_DESCRIPTION,
)

##### 3. Verify MLflow registry target

In [0]:
##### For Unity Catalog:

mlflow.set_registry_uri("databricks-uc")

print(
    "Registry URI:",
    mlflow.get_registry_uri()
)

Databricks recommends using Unity Catalog Model Registry, and databricks-uc explicitly targets it.

Even though MLflow 3 can default to Unity Catalog in many Databricks workspaces, still like this explicit line in a production notebook because it makes our intent obvious.

##### 4. Why Notebook 13 should not depend on the model_uri Python variable from Notebook 12

This is important because of your project rule: Every notebook must be independently runnable.

Notebook 12 had: model_uri in memory.

Notebook 13 must not assume that variable exists.

Instead, Notebook 13 should query MLflow and identify the model associated with the successful: 
minilm_full_finetuning run.

That gives us:

``` text

Persistent MLflow state
        ↓
Notebook 13 discovers model
        ↓
register model

instead of:

Notebook 12 Python variable
        ↓
Notebook 13

```

which would violate notebook independence.

##### 5. Find the latest successful training run

In [0]:
experiment = (
    mlflow.get_experiment_by_name(
        MLFLOW_EXPERIMENT_NAME
    )
)

if experiment is None:
    raise ValueError(
        f"MLflow experiment not found: "
        f"{MLFLOW_EXPERIMENT_NAME}"
    )

print(
    "Experiment ID:",
    experiment.experiment_id
)

print(MLFLOW_EXPERIMENT_NAME)

In [0]:
#this part dynamically selects the run:
runs_df = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ],
    filter_string=(
        f"tags.mlflow.runName = "
        f"'{MLFLOW_RUN_NAME}'"
    ),
    order_by=[
        "start_time DESC"
    ],
    max_results=10,
)

display(
    runs_df[
        [
            "run_id",
            "status",
            "start_time",
            "metrics.test_accuracy",
            "metrics.test_macro_f1",
        ]
    ]
)

In [0]:
print(MLFLOW_RUN_NAME)

In [0]:
successful_runs_df = (
    runs_df[
        runs_df["status"]
        == "FINISHED"
    ]
    .reset_index(
        drop=True
    )
)

if successful_runs_df.empty:
    raise ValueError(
        "No successful "
        f"'{MLFLOW_RUN_NAME}' "
        "run was found."
    )

source_run_id = (
    successful_runs_df
    .iloc[0]["run_id"]
)

print(
    "Selected Run ID:",
    source_run_id
)

##### 6. Find the Logged Model associated with that run

In [0]:
client = MlflowClient()

logged_models = client.search_logged_models(
    experiment_ids=[
        experiment.experiment_id
    ]
)

matching_models = [
    logged_model
    for logged_model in logged_models
    if (
        logged_model.source_run_id
        == source_run_id
        and logged_model.name == "model"
        and logged_model.status == "READY"
    )
]

if not matching_models:
    raise ValueError(
        "No READY Logged Model named 'model' "
        f"was found for source run "
        f"{source_run_id}."
    )

logged_model = matching_models[0]

source_model_uri = (
    f"models:/{logged_model.model_id}"
)

print("Model ID:", logged_model.model_id)
print("Model Name:", logged_model.name)
print("Model Status:", logged_model.status)
print("Source Run ID:", logged_model.source_run_id)
print("Source Model URI:", source_model_uri)

##### 7. Verify model signature before registration

In [0]:
source_model_uri = (
    f"models:/{logged_model.model_id}"
)

print(
    "Selected Model ID:",
    logged_model.model_id
)

print(
    "Source Run ID:",
    logged_model.source_run_id
)

print(
    "Source Model URI:",
    source_model_uri
)

In [0]:
print(experiment)

In [0]:
print(logged_models)

In [0]:
model_info = mlflow.models.get_model_info(
    source_model_uri
)

print(
    "Model URI:",
    model_info.model_uri
)

print(
    "Signature:",
    model_info.signature
)

In [0]:
if model_info.signature is None:
    raise ValueError(
        "The logged model has no "
        "MLflow signature and should "
        "not be registered."
    )

##### 8. Register the model in Unity Catalog

In [0]:
registration_result = (
    mlflow.register_model(
        model_uri=source_model_uri,
        name=REGISTERED_MODEL_NAME,
    )
)

MLflow will create the registered model if it does not yet exist; otherwise it creates a new version under the existing model.

In [0]:
print(
    "Registered model:",
    registration_result.name
)

print(
    "Version:",
    registration_result.version
)

print(
    "Status:",
    registration_result.status
)

In [0]:
registered_version = str(
    registration_result.version
)

##### 9. Add model description


In [0]:
client = MlflowClient(
    registry_uri="databricks-uc"
)

In [0]:
client.update_registered_model(
    name=REGISTERED_MODEL_NAME,
    description=(
        REGISTERED_MODEL_DESCRIPTION
    ),
)

In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Candidate",
    version=registered_version,
)

print(
    f"Alias 'Candidate' assigned to "
    f"{REGISTERED_MODEL_NAME} "
    f"version {registered_version}."
)

##### 10. Add version description

In [0]:
version_description = (f"""Full fine-tuning of sentence-transformers/all-MiniLM-L6-v2 for Billing, Cancellation, Login, and Technical support-ticket classification. The model accepts raw ticket_text through                       
    an MLflow PyFunc inference interface."""
)

client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=registered_version,
    description=version_description,
)

Registered model description → What is this model generally?

Version description → What is special about this particular version?

##### 11. Add useful model-version tags

In [0]:
version_tags = {
    "project":
        "support_ticket_nlp",

    "model_family":
        "MiniLM",

    "task":
        "multiclass_classification",

    "fine_tuning":
        "full",

    "input_type":
        "raw_text",

    "num_classes":
        str(
            len(CLASS_NAMES)
        ),

    "source_run_id":
        source_run_id,
}

In [0]:
for key, value in version_tags.items():

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=registered_version,
        key=key,
        value=value,
    )

##### 12. Retrieve the registered version

In [0]:
registered_model_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=registered_version,
    )
)

print(
    "Name:",
    registered_model_version.name
)

print(
    "Version:",
    registered_model_version.version
)

print(
    "Status:",
    registered_model_version.status
)

print(
    "Source:",
    registered_model_version.source
)

##### 13. Build registered-model URI

In [0]:
registered_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}/"
    f"{registered_version}"
)

print(
    "Registered Model URI:",
    registered_model_uri
)

##### 14. Load the registered model

In [0]:
registered_model = (
    mlflow.pyfunc.load_model(
        registered_model_uri
    )
)

Notebook 12: 

models:/m-xxxx means: Load this specific MLflow Logged Model.

Notebook 13:

models:/catalog.schema.model/version means: Load this version from the governed Unity Catalog Model Registry.

##### 15. Raw-text inference test

In [0]:
registration_test_input = pd.DataFrame(
    {
        TEXT_COL: [
            (
                "Account locked "
                "after many attempts"
            ),
            (
                "My internet connection "
                "keeps dropping"
            ),
            (
                "Please cancel "
                "my subscription"
            ),
            (
                "I was charged twice "
                "this month"
            ),
        ]
    }
)

In [0]:
registration_test_output = (
    registered_model.predict(
        registration_test_input
    )
)

display(
    registration_test_output
)

##### 16. Verify inference contract

In [0]:
assert (
    len(registration_test_output)
    ==
    len(registration_test_input)
)

assert (
    "predicted_category"
    in registration_test_output.columns
)

assert (
    "confidence"
    in registration_test_output.columns
)

assert set(
    registration_test_output[
        "predicted_category"
    ]
).issubset(
    set(CLASS_NAMES)
)

assert (
    registration_test_output[
        "confidence"
    ]
    .between(
        0.0,
        1.0,
    )
    .all()
)

print(
    "Unity Catalog registered "
    "model verification passed."
)

##### 17. Final registration summary

In [0]:
registration_summary_df = pd.DataFrame(
    [
        {
            "registered_model":
                REGISTERED_MODEL_NAME,

            "version":
                registered_version,

            "source_run_id":
                source_run_id,

            "source_model_uri":
                source_model_uri,

            "registered_model_uri":
                registered_model_uri,

            "num_classes":
                len(CLASS_NAMES),

            "input":
                TEXT_COL,
        }
    ]
)

display(
    registration_summary_df
)

In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Candidate",
    version=registered_version,
)

print(
    f"Alias 'Candidate' assigned to "
    f"{REGISTERED_MODEL_NAME} "
    f"version {registered_version}."
)

In [0]:
candidate_model = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias="Candidate",
    )
)

print(
    "Candidate version:",
    candidate_model.version
)

``` text

Logged Model
    ↓
Registered Model Version 1
    ↓
Alias = Candidate
    ↓
Deploy
    ↓
Endpoint testing
    ↓
Promote to Champion

```

Notebook 13
→ register Version 1
→ alias Candidate

Notebook 14
→ deploy Candidate

Notebook 15
→ endpoint testing

after successful validation
→ promote Candidate → Champion

In [0]:
candidate_uri = (
    f"models:/{REGISTERED_MODEL_NAME}@Candidate"
)

##### What Notebook 13 teaches

The important distinction is now:

``` text

MLflow Experiment
        ↓
tracks training work

MLflow Logged Model
        ↓
stores one packaged model artifact

Unity Catalog Registered Model
        ↓
governed logical model

Model Version
        ↓
specific immutable registered build

```

``` text

Experiment
/Users/.../support_ticket_nlp_modeling

        ↓

Run
minilm_full_finetuning

        ↓

Logged Model
models:/m-...

        ↓

Registered Model
dbw_agentic_ai_dev
.support_ticket_ai
.support_ticket_transformer_classifier

        ↓

Version 1

```

This is an important MLOps distinction.

Notebook 12 answered: What exactly did we train and package?

Notebook 13 answers: Which packaged model are we approving as a governed model asset?


TABLE
→ model selection mechanism

JSON MANIFEST
→ model selection mechanism

mlflow.register_model(...)
→ actual registration

mlflow.pyfunc.load_model(...)
→ load/verify registered model

ML / NN projects
- explicit selection artifact
- (table or manifest)
- selected run/model
- register

Current Transformer project
- dynamically discover latest
- successful intended training run
- find its Logged Model
- register

##### Key Learnings

- Registration does not retrain the model. We register the already validated Logged Model produced by Notebook 12.

- A Logged Model and Registered Model are not the same thing. The Logged Model is an MLflow artifact/build. The Unity Catalog registered model provides governance, naming, versioning, access control and lifecycle management.

- Unity Catalog models use three-level names : catalog.schema.model

- Our model uses: dbw_agentic_ai_dev.support_ticket_ai.support_ticket_transformer_classifier

- Every registration creates a model version. Re-registering another validated build later may produce Version 2, Version 3, and so forth.

- A model signature is required for Unity Catalog registration. Notebook 12 already logged the signature, which is one reason we included it in the production MLflow package.

- Registration is not complete until loading is verified. Notebook 13 reloads the model through its Unity Catalog URI and performs raw-text inference.

##### Conclusion

A suitable final conclusion is:

- Notebook 13 promoted the validated MLflow Logged Model from Notebook 12 into the Unity Catalog Model Registry. The model was registered using the governed three-level Unity Catalog name dbw_agentic_ai_dev.support_ticket_ai.support_ticket_transformer_classifier, creating a versioned model asset without retraining the Transformer.

- Model and version descriptions and governance tags were added to preserve the model's purpose, task, fine-tuning strategy and source run lineage. The registered version was then loaded through its Unity Catalog model URI and successfully used for raw-text support-ticket inference.

- This separates model experimentation from model governance: MLflow experiments record how the model was produced, while Unity Catalog provides the controlled, versioned model asset that can subsequently be deployed.